```
Giới thiệu
----------
Trong báo cáo này, chúng tôi trình bày quy trình xây dựng một mô hình nhận diện đối tượng (Object Detection) cho các phương tiện giao thông tại Việt Nam. Mô hình nền (backbone) được sử dụng là SSD MobileNetV3 đã được huấn luyện sẵn (pretrained) trên tập dữ liệu COCO. Tiếp theo, mô hình được tinh chỉnh (finetune) trên tập dữ liệu phương tiện giao thông Việt Nam nhằm tối ưu khả năng nhận diện theo bối cảnh thực tế tại Việt Nam.

Dữ liệu
-------
Tập dữ liệu được thu thập từ nền tảng Roboflow với tổng cộng khoảng 2000 ảnh, bao gồm bốn lớp đối tượng:
- Motorcycle (xe máy)
- Car (xe hơi)
- Truck (xe tải)
- Bus (xe buýt)

Tập dữ liệu được gán nhãn theo định dạng COCO JSON và chia theo tỉ lệ 80-10-10 cho train, validation và test.

Phương pháp
-----------
Quy trình xây dựng mô hình bao gồm bốn giai đoạn chính.

1. Su dung mo hinh pretrained SSD MobileNetV3
   - Mo hinh SSD MobileNetV3 duoc train san tren COCO.
   - Su dung pretrained giup hoi tu nhanh, do chinh xac tot hon.

2. Finetune tren tap phuong tien giao thong Viet Nam
   - Thay doi detection head de phu hop 4 classes.
   - Adjustment: learning rate nho, data augmentation manh, early stopping de tranh overfitting.

3. Pruning mo hinh
   - Ap dung structured pruning (giam kenh convolution).
   - Muc tieu: giam kich thuoc model, giam FLOPs, va khong giam accuracy qua 2%.

4. Quantization INT8
   - Thuc hien post-training quantization.
   - Muc tieu: giam kich thuoc 3-4 lan, tang toc CPU, accuracy giam < 2%.

Ket qua
-------
Cac chi tieu danh gia:
- mAP tren tap test
- Kich thuoc mo hinh truoc/sau pruning + quantization
- FPS khi chay tren CPU

Ket qua dat duoc:
- Accuracy giam < 2%
- FPS > 30 khi chay tren CPU
- Model INT8 nhe, phu hop edge devices

Ket luan
--------
Pipeline finetune -> pruning -> quantization tren SSD MobileNetV3 tao ra mo hinh nhe, nhanh, do chinh xac tot, phu hop trien khai tren camera giao thong, thiet bi nhung, hoac CPU cong suat thap.
```


### Vì sao sử dụng thư viện `torch-pruning`?

Trong TensorFlow, các công cụ pruning được hỗ trợ tốt chủ yếu cho mô hình **classification** (ví dụ các model Keras như ResNet, EfficientNet, v.v.). Tuy nhiên, đối với các mô hình **object detection** (SSD, Faster R-CNN, RetinaNet, v.v.), TensorFlow **không có sẵn** pipeline pruning hoàn chỉnh. Nếu muốn pruning detection model trong TensorFlow, thường phải:
- Tự viết lại logic pruning cho từng layer / nhánh,
- Xử lý thủ công các kết nối phức tạp (FPN, skip-connection, multi-head prediction),
- Dễ phát sinh lỗi kiến trúc và rất tốn thời gian debug.

Để tránh việc phải “custom quá nhiều” trong TensorFlow, một hướng thực tế hơn là chuyển sang **PyTorch** và sử dụng thư viện chuyên cho structural pruning trên các kiến trúc phức tạp. Một trong những thư viện nổi bật là **`torch-pruning`**.

Thư viện `torch-pruning` được xây dựng dựa trên ý tưởng **Dependency Graph (DepGraph)** – một thuật toán đồ thị để mô hình hóa quan hệ phụ thuộc giữa các layer và nhóm các tham số phải được prune cùng nhau. Cách tiếp cận này cho phép:
- Pruning có cấu trúc (structured pruning) theo **kênh / filter** thay vì chỉ zero-weight,
- Giữ kiến trúc mạng hợp lệ sau khi cắt bớt kênh, kể cả với mạng có skip-connection, FPN,
- Áp dụng được cho nhiều loại mô hình khác nhau, bao gồm cả **SSD, Faster R-CNN, YOLO**, v.v.

`torch-pruning` được phát triển dựa trên bài báo:

> **DepGraph: Towards Any Structural Pruning**  
> Gongfan Fang, Xinyin Ma, Mingli Song, Michael Bi Mi, Xinchao Wang  
> CVPR 2023

Nói ngắn gọn:
- TensorFlow: pruning object detection phải tự custom nhiều, khó và dễ lỗi.
- PyTorch + `torch-pruning`: có sẵn công cụ dựa trên DepGraph, hỗ trợ structured pruning cho các mô hình detection như SSD MobileNetV3 sau khi đã finetune trên dữ liệu phương tiện giao thông Việt Nam.

In [1]:
!pip install torch-pruning --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 88.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 66.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 74.4 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvi

### Khởi tạo mô hình SSD MobileNetV3 (custom số lớp)

Để huấn luyện mô hình SSD MobileNetV3 trên bộ dữ liệu mới, ta cần thay đổi phần detection head để phù hợp với số lượng lớp (classes) mong muốn. Quy trình khởi tạo mô hình gồm các bước:

1. **Tải mô hình SSD MobileNetV3 (pretrained COCO)**  
   Sử dụng trọng số `COCO_V1` có sẵn từ torchvision.  
   Điều này giúp tận dụng đặc trưng đã học từ COCO và tăng tốc quá trình huấn luyện.

2. **Trích xuất thông tin kiến trúc từ backbone**  
   - Số lượng kênh đầu ra từ backbone (in_channels).  
   - Số anchors trên mỗi location (num_anchors).  
   Đây là tham số quan trọng để tạo lại detection head.

3. **Xây dựng lại detection head**  
   - Dùng lớp `SSDLiteHead` của torchvision.  
   - Truyền vào số lớp mới (`num_classes`).  
   - Giữ nguyên cấu trúc anchor generator và FPN như bản pretrained.

4. **Trả về mô hình hoàn chỉnh**  
   Mô hình mới có thể huấn luyện trực tiếp với dataset có số lớp mong muốn (4 hoặc 6 class).



In [ ]:
import torch
import torch.nn as nn
from functools import partial

from torchvision.models.detection.ssdlite import (
    ssdlite320_mobilenet_v3_large,
    SSDLite320_MobileNet_V3_Large_Weights,
    SSDLiteHead,
)
from torchvision.models.detection import _utils as det_utils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def get_ssd_model(num_classes: int):
    """
    Creates an SSD MobileNetV3 model with a custom number of classes (num_classes).
    Steps include:
    - Loading pretrained COCO weights
    - Retrieving backbone information
    - Recreating the detection head corresponding to the new number of classes
    """

    # 1. Load model pretrained on COCO
    weights = SSDLite320_MobileNet_V3_Large_Weights.COCO_V1
    model = ssdlite320_mobilenet_v3_large(weights=weights)

    # 2. Get output channels from the backbone (using torchvision's default utility function)
    in_channels = det_utils.retrieve_out_channels(model.backbone, (320, 320))

    # 3. Get the number of anchors per feature map location
    num_anchors = model.anchor_generator.num_anchors_per_location()

    # 4. Standardize BatchNorm similar to the original SSDLite setup
    norm_layer = partial(nn.BatchNorm2d, eps=0.001, momentum=0.03)

    # 5. Recreate the detection head to match the new number of classes
    #    (replaces the old head - 91 COCO classes)
    model.head = SSDLiteHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=num_classes,
        norm_layer=norm_layer
    )

    print("Created SSD MobileNetV3 model with NUM_CLASSES =", num_classes)
    return model
    

# Example: create a 4-class model
NUM_CLASSES = 4
model = get_ssd_model(NUM_CLASSES).to(device)

✔ Tạo model SSD MobileNetV3 với NUM_CLASSES = 6


### Hàm `load_coco_annotations` – Tiền xử lý dataset COCO

Hàm này dùng để đọc annotation theo chuẩn COCO từ thư mục `train/`, `valid/` hoặc `test/` của Roboflow.  
Mục tiêu chính:

1. **Đọc file `_annotations.coco.json`**  
   Đây là file chứa toàn bộ metadata: danh sách ảnh, bbox, nhãn, category.

2. **Chuẩn hóa lại nhãn**  
   - COCO category_id không liên tục, ví dụ 1, 3, 5, 7  
   - Ta ánh xạ về nhãn liên tục `[1..K]` để phù hợp với PyTorch detection.

3. **Tạo record cho từng ảnh**  
   Mỗi record gồm:
   - `file_name`: tên file ảnh  
   - `boxes`: danh sách bbox `[xmin, ymin, xmax, ymax]`  
   - `labels`: danh sách nhãn tương ứng  

4. **Ghép annotation vào từng ảnh**  
   Với mỗi annotation trong JSON:
   - Lấy bbox dạng COCO `[x, y, width, height]`  
   - Chuyển sang format `[xmin, ymin, xmax, ymax]`  
   - Đưa vào đúng ảnh dựa theo `image_id`

Hàm trả về:
- `records`: danh sách dict, mỗi dict tương ứng 1 ảnh  
- `cat_id_to_label`: ánh xạ từ COCO category_id → nhãn liên tục  
- `label_to_name`: map từ nhãn → tên lớp (dùng để visualize)


In [ ]:
import os
import json
import torch

def load_coco_annotations(split_dir):
    """
    split_dir: path to the train/ or valid/ or test/ directory
               (this directory contains _annotations.coco.json + images)

    Returns:
      - records: list of dicts describing each image
      - cat_id_to_label: mapping from category_id in COCO -> continuous label 1..K
      - label_to_name: mapping from label -> class name (used for debug/visualization)
    """

    # 1. Read the COCO JSON annotation file
    ann_path = os.path.join(split_dir, "_annotations.coco.json")
    with open(ann_path, "r", encoding="utf-8") as f:
        coco = json.load(f)

    # Extract primary fields
    images = coco["images"]             # List of images
    annotations = coco["annotations"]   # List of bounding boxes
    categories = coco["categories"]     # List of classes

    # 2. Create map: COCO category_id -> continuous label [1..K]
    # (COCO IDs are often non-contiguous, so normalization is required)
    cat_ids = [c["id"] for c in categories]
    cat_ids_sorted = sorted(cat_ids)
    cat_id_to_label = {cid: (i + 1) for i, cid in enumerate(cat_ids_sorted)}

    # Reverse map: label -> class name
    label_to_name = {
        cat_id_to_label[c["id"]]: c["name"]
        for c in categories
    }

    # 3. Initialize records for each image (empty boxes + labels)
    id_to_rec = {}
    for img in images:
        img_id = img["id"]

        # Roboflow sometimes uses "file_name", sometimes uses "name"
        file_name = img.get("file_name", img.get("name"))

        id_to_rec[img_id] = {
            "file_name": file_name,
            "boxes": [],
            "labels": [],
        }

    # 4. Attach bounding boxes and labels to the correct image
    for ann in annotations:
        img_id = ann["image_id"]
        if img_id not in id_to_rec:
            # Avoid errors for unmatched annotations
            continue

        # COCO bbox: [x, y, w, h]
        x, y, w, h = ann["bbox"]
        xmin = x
        ymin = y
        xmax = x + w
        ymax = y + h

        # Map category_id -> continuous label
        cat_id = ann["category_id"]
        label = cat_id_to_label[cat_id]

        id_to_rec[img_id]["boxes"].append([xmin, ymin, xmax, ymax])
        id_to_rec[img_id]["labels"].append(label)

    # Convert from dict -> list
    records = list(id_to_rec.values())
    return records, cat_id_to_label, label_to_name

### Lớp `COCORoboflowSSDDataset` – Dataset cho SSD (PyTorch)

Lớp này là một `torch.utils.data.Dataset` custom, dùng để đọc dữ liệu theo chuẩn COCO (xuất từ Roboflow) cho bài toán object detection với SSD MobileNetV3.

Các ý chính:

1. **Khởi tạo (`__init__`)**
   - Tham số `split_dir`: đường dẫn tới thư mục `train/`, `valid/` hoặc `test/`.
   - Gọi lại hàm `load_coco_annotations(split_dir)` để:
     - Đọc file `_annotations.coco.json`
     - Tạo `records`: danh sách ảnh + bbox + labels
     - Lấy map `cat_id_to_label` và `label_to_name` để dùng khi cần.

2. **Số lượng phần tử (`__len__`)**
   - Trả về số lượng ảnh trong tập (dùng cho DataLoader).

3. **Lấy 1 mẫu (`__getitem__`)**
   - Lấy record tương ứng với index `idx`.
   - Ghép đường dẫn ảnh: `split_dir/file_name`.
   - Đọc ảnh bằng `PIL.Image` và convert sang RGB.
   - Lấy danh sách bounding boxes và labels từ `records`.

4. **Xử lý trường hợp ảnh không có bounding box**
   - Nếu `boxes_list` rỗng:
     - Tạo tensor `boxes` kích thước `(0, 4)`.
     - Tạo `labels` kích thước `(0,)`.
   - Nếu có bbox:
     - Chuyển list -> tensor `torch.float32` và `torch.int64`.

5. **Tạo dictionary `target`**
   - `boxes`: tensor `[N, 4]` theo format `[xmin, ymin, xmax, ymax]`
   - `labels`: tensor `[N]` – nhãn integer
   - `image_id`: id ảnh (ở đây dùng index `idx`)
   - `area`: diện tích từng bbox (dùng cho mAP COCO)
   - `iscrowd`: tensor zeros (mặc định không có crowd annotation)

6. **Transforms**
   - Nếu có truyền `self.transforms`, áp dụng lên ảnh trước khi trả về.
   - Dataset trả về: `(img, target)` phù hợp với API detection của torchvision.


In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import os
import torch

class COCORoboflowSSDDataset(Dataset):
    """
    Custom Dataset for SSD, reading annotations in COCO format (Roboflow export).
    Each returned element:
      - img   : Image Tensor (C, H, W)
      - target: dict containing boxes, labels, etc.
    """

    def __init__(self, split_dir, transforms=None):
        """
        split_dir: directory containing images + the _annotations.coco.json file
                   e.g., "./data/train", "./data/valid"
        transforms: transformations (augmentation / normalize) applied to the image
        """
        self.split_dir = split_dir
        self.transforms = transforms

        # Read all COCO annotations -> records + class mapping
        self.records, self.cat_id_to_label, self.label_to_name = load_coco_annotations(split_dir)

    def __len__(self):
        # Number of samples in the dataset = number of images
        return len(self.records)

    def __getitem__(self, idx):
        # Get the record corresponding to the index
        rec = self.records[idx]

        # Create the image file path
        img_path = os.path.join(self.split_dir, rec["file_name"])

        # Read the image and convert to RGB
        img = Image.open(img_path).convert("RGB")

        # Get list of bboxes and labels from the record
        boxes_list = rec["boxes"]
        labels_list = rec["labels"]

        # Case where the image has no bounding boxes
        if len(boxes_list) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            # Convert list -> tensor
            boxes = torch.as_tensor(boxes_list, dtype=torch.float32).reshape(-1, 4)
            labels = torch.as_tensor(labels_list, dtype=torch.int64)

        # Create target dict in the standard torchvision detection format
        target = {
            "boxes": boxes,                    # [N, 4]
            "labels": labels,                  # [N]
            "image_id": torch.tensor([idx]),   # image id (using idx is sufficient)
        }

        # Calculate bounding box area (used for COCO metrics)
        if boxes.numel() > 0:
            area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        else:
            area = torch.zeros((0,), dtype=torch.float32)

        target["area"] = area
        # iscrowd = 0: default to not using crowd annotation
        target["iscrowd"] = torch.zeros((boxes.shape[0],), dtype=torch.int64)

        # Apply transforms if available (only applied to the image, target remains unchanged)
        if self.transforms is not None:
            img = self.transforms(img)

        return img, target

### Tạo transforms, Dataset và DataLoader cho SSD MobileNetV3

Đối với mô hình SSDLite (SSD MobileNetV3), backbone đã được thiết kế để xử lý ảnh trực tiếp ở kích thước gốc sau khi đưa qua transform nội bộ. Vì vậy trong giai đoạn load dữ liệu, ta chỉ cần:

- Chuyển ảnh sang Tensor (`T.ToTensor()`)
- Không cần resize hoặc padding thủ công
- Không cần normalization vì SSDLite sử dụng chuẩn normalize nội bộ

Sau khi tạo `Dataset` từ COCO Roboflow, cần định nghĩa `collate_fn` để ghép batch theo đúng format của PyTorch detection models. Đặc điểm:

- Ảnh có thể khác kích thước → không thể stack trực tiếp  
- `collate_fn` ghép thành tuple:  
  `([img1, img2, ...], [target1, target2, ...])`

DataLoader cuối cùng được tạo cho `train`, `valid`, `test`, với:

- `batch_size = 32`
- `num_workers = 4`
- `shuffle=True` cho train
- `collate_fn` tùy chỉnh cho detection


In [ ]:
import torchvision.transforms as T
import os
from torch.utils.data import DataLoader

# Dataset path following Roboflow COCO export structure
root_dir = "/kaggle/input/datasetcantho"

train_dir = os.path.join(root_dir, "train")
valid_dir = os.path.join(root_dir, "valid")
test_dir = os.path.join(root_dir, "test")

# ------------------------------------------------------------
# Transforms: SSD does not require manual resizing,
# only ToTensor() is needed as the model will handle scaling internally.
# ------------------------------------------------------------
train_transforms = T.Compose([
    T.ToTensor(),                 # convert image to tensor [0..1]
])

valid_transforms = T.Compose([
    T.ToTensor(),
])

# ------------------------------------------------------------
# Create Dataset for train and valid
# ------------------------------------------------------------
train_dataset = COCORoboflowSSDDataset(train_dir, transforms=train_transforms)
valid_dataset = COCORoboflowSSDDataset(valid_dir, transforms=valid_transforms)

# ------------------------------------------------------------
# collate_fn: extremely important for object detection!
# It should not stack tensors by batch like image classification,
# but must group each sample into a tuple(images, targets).
# ------------------------------------------------------------
def collate_fn(batch):
    # batch is a list of [(img, target), (img, target), ...]
    return tuple(zip(*batch))

# ------------------------------------------------------------
# Create DataLoaders for train/valid/test
# ------------------------------------------------------------
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    collate_fn=collate_fn,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    collate_fn=collate_fn,
)

test_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    collate_fn=collate_fn,
)

### Huấn luyện SSD MobileNetV3 trên dataset phương tiện giao thông

Sau khi đã xây dựng model, Dataset và DataLoader, bước tiếp theo là thiết lập **optimizer** và **vòng lặp huấn luyện (training loop)**.

Các ý chính:

1. **Chọn tham số cần tối ưu**
   - Lấy tất cả các tham số `p` trong `model.parameters()` với `p.requires_grad == True`.
   - Điều này giúp tránh tối ưu các tham số bị đóng băng (nếu có).

2. **Optimizer**
   - Dùng `Adam` với:
     - `lr = 1e-4` (learning rate nhỏ để finetune từ pretrained COCO)
     - `weight_decay = 1e-5` (L2 regularization nhẹ để tránh overfitting)

3. **Vòng lặp huấn luyện**
   - Số epoch: `num_epochs = 40` (có thể điều chỉnh tùy theo dataset).
   - Mỗi epoch:
     - `model.train()` để bật chế độ training.
     - Duyệt từng batch từ `train_loader`.
     - Đưa `images`, `targets` lên `device` (CPU/GPU).
     - Gọi `model(images, targets)`:
       - Trả về `loss_dict` gồm nhiều thành phần loss (classification, localization, regularization, v.v.)
     - Tổng hợp loss: `losses = sum(loss for loss in loss_dict.values())`
     - Backprop:
       - `losses.backward()`
       - Dùng `clip_grad_norm_` để tránh gradient bùng nổ.
     - `optimizer.step()` để cập nhật trọng số.
   - Cuối mỗi epoch in ra `avg_loss` để theo dõi.

4. **Lưu model**
   - Lưu `state_dict` (trọng số thuần): `"ssdlite320_mbv3_finetuned_state.pth"`
   - Lưu toàn bộ model (cả kiến trúc + trọng số): `"ssdlite320_mbv3_finetuned_full.pth"`
   - File `*_full.pth` tiện dùng để load trực tiếp khi suy luận hoặc pruning.


In [ ]:
import torch.optim as optim
import torch

# ------------------------------------------------------------
# Get parameters that require optimization (requires_grad = True)
# ------------------------------------------------------------
params = [p for p in model.parameters() if p.requires_grad]

# ------------------------------------------------------------
# Initialize Adam optimizer for the entire model
# ------------------------------------------------------------
optimizer = optim.Adam(
    params,
    lr=1e-4,              # small learning rate for finetuning from pretrained COCO
    weight_decay=1e-5,    # light L2 regularization
)

num_epochs = 40

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    # Iterate through each batch in train_loader
    for images, targets in train_loader:
        # Move images and targets to the corresponding GPU/CPU device
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Zero the old gradients
        optimizer.zero_grad()

        # Call the model in training mode:
        # For detection models in torchvision,
        # model(images, targets) returns a loss_dict
        loss_dict = model(images, targets)

        # Aggregate all losses into a single scalar
        losses = sum(loss for loss in loss_dict.values())

        # Backpropagation
        losses.backward()

        # Gradient clipping (to prevent excessive gradients)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)

        # Update weights
        optimizer.step()

        # Accumulate loss to calculate the average loss per epoch
        running_loss += losses.item()

    avg_loss = running_loss / len(train_loader)
    print(f"[Epoch {epoch+1}/{num_epochs}] loss: {avg_loss:.4f}")

# ------------------------------------------------------------
# Save the model after finetuning
# ------------------------------------------------------------
# Save only the weights (state_dict)
torch.save(model.state_dict(), "ssdlite320_mbv3_finetuned_state.pth")

# Save the full model (architecture + weights) - convenient for pruning/quantization
torch.save(model, "ssdlite320_mbv3_finetuned_full.pth")

[Epoch 1/40] loss: 11.2775
[Epoch 2/40] loss: 6.3657
[Epoch 3/40] loss: 4.1290
[Epoch 4/40] loss: 3.0827
[Epoch 5/40] loss: 2.5036
[Epoch 6/40] loss: 2.1441
[Epoch 7/40] loss: 1.8796
[Epoch 8/40] loss: 1.6851
[Epoch 9/40] loss: 1.5362
[Epoch 10/40] loss: 1.3990
[Epoch 11/40] loss: 1.2738
[Epoch 12/40] loss: 1.1656
[Epoch 13/40] loss: 1.0673
[Epoch 14/40] loss: 0.9712
[Epoch 15/40] loss: 0.8793
[Epoch 16/40] loss: 0.7951
[Epoch 17/40] loss: 0.7122
[Epoch 18/40] loss: 0.6533
[Epoch 19/40] loss: 0.5783
[Epoch 20/40] loss: 0.5144
[Epoch 21/40] loss: 0.4589
[Epoch 22/40] loss: 0.4096
[Epoch 23/40] loss: 0.3584
[Epoch 24/40] loss: 0.3216
[Epoch 25/40] loss: 0.2830
[Epoch 26/40] loss: 0.2573
[Epoch 27/40] loss: 0.2283
[Epoch 28/40] loss: 0.2056
[Epoch 29/40] loss: 0.1848
[Epoch 30/40] loss: 0.1665
[Epoch 31/40] loss: 0.1523
[Epoch 32/40] loss: 0.1385
[Epoch 33/40] loss: 0.1253
[Epoch 34/40] loss: 0.1179
[Epoch 35/40] loss: 0.1151
[Epoch 36/40] loss: 0.1008
[Epoch 37/40] loss: 0.0943
[Epoch 38

### Đánh giá mAP@0.5 trên tập test bằng TorchMetrics

Sau khi huấn luyện xong SSD MobileNetV3, ta đánh giá chất lượng mô hình trên tập test bằng `MeanAveragePrecision` (torchmetrics) với thang đo **mAP@0.5**.

Kết quả thu được:

- **mAP@0.5 ≈ 0.66**

Giá trị này chưa cao, nguyên nhân chính có thể đến từ:

1. **Mất cân bằng lớp (class imbalance)**  
   - Một số lớp (ví dụ: xe máy, xe hơi) xuất hiện rất nhiều.  
   - Các lớp khác (xe tải, xe buýt) xuất hiện ít hơn, dẫn đến model học thiên lệch.

2. **Đối tượng nhỏ so với background**  
   - Nhiều phương tiện chiếm diện tích rất nhỏ trong khung hình (đứng xa camera, bị che khuất, v.v.).  
   - SSD MobileNetV3 vốn là kiến trúc nhẹ, nên khả năng bắt các object rất nhỏ (small objects) còn hạn chế, đặc biệt khi chỉ dùng cấu hình mặc định.

3. **Thiết lập huấn luyện còn đơn giản**  
   - Chỉ dùng transform cơ bản (`ToTensor()`), chưa khai thác nhiều kỹ thuật tăng cường dữ liệu (augmentation).  
   - Số epoch, learning rate, batch size… đang để ở mức tương đối “an toàn”, không tối ưu cho từng lớp.

Để cải thiện mAP trong thực tế, có thể áp dụng các kỹ thuật sau:

- **Thay đổi cách xử lý ảnh**  
  - Huấn luyện với **nhiều kích thước ảnh** (multi-scale training).  
  - Chia ảnh thành các patch nhỏ hơn, **zoom** vùng chứa object rồi đem train.  

- **Tăng cường dữ liệu (data augmentation)**  
  - Random crop, random zoom, mosaic/cutout, photometric distortion, v.v.  
  - Tập trung tăng cường các mẫu chứa object nhỏ, các lớp hiếm.

- **Xử lý mất cân bằng**  
  - Dùng **weighted sampler / weighted DataLoader** để oversample các lớp ít xuất hiện.  
  - Áp dụng **hàm loss phù hợp hơn**, ví dụ: **Focal Loss** để giảm ảnh hưởng của negative dễ.

- **Điều chỉnh hoặc thay đổi kiến trúc**  
  - Dùng phiên bản backbone mạnh hơn hoặc FPN tốt hơn cho small objects.  
  - Điều chỉnh anchor sizes, aspect ratios phù hợp với kích thước phương tiện trong dữ liệu Việt Nam.

Tuy nhiên, các kỹ thuật tối ưu sâu hơn này:

- Đòi hỏi nhiều thời gian thử nghiệm, tính toán lại hyperparameters.  
- Có thể vượt ngoài phạm vi nội dung và mục tiêu của môn học hiện tại.

Vì vậy, trong phạm vi đồ án/môn học này, nhóm **chỉ sử dụng cấu hình mặc định** (pretrained SSD MobileNetV3 + finetune cơ bản) và ghi nhận kết quả **mAP@0.5 ≈ 0.66** như một baseline cho bài toán nhận diện phương tiện giao thông Việt Nam.

In [ ]:
import torch
from torchmetrics.detection.mean_ap import MeanAveragePrecision

device = torch.device("cpu")  # Force to CPU to avoid anchor/op errors on GPU

# ------------------------------------------------------------
# Load the full finetuned model (architecture + weights)
# ------------------------------------------------------------
model = torch.load(
    "/kaggle/working/ssdlite320_mbv3_finetuned_full.pth",
    map_location=device,
    weights_only=False,  # very important for PyTorch >= 2.6
)

model.to(device)
model.eval()
print("Model loaded on:", device)

# ------------------------------------------------------------
# Initialize mAP metric from torchmetrics
# ------------------------------------------------------------
metric = MeanAveragePrecision(
    iou_type="bbox",
    iou_thresholds=[0.5],  # mAP@0.5
)

# ------------------------------------------------------------
# Evaluation loop over test_loader
# ------------------------------------------------------------
with torch.no_grad():
    for images, targets in test_loader:
        # images is list[Tensor[C,H,W]]
        images = [img.to(device) for img in images]

        # Run the model in inference mode
        # outputs: list[ {boxes, labels, scores} ]
        outputs = model(images)

        # Normalize to the format required by torchmetrics
        preds = []
        gts = []

        for out, tgt in zip(outputs, targets):
            # Model predictions
            preds.append({
                "boxes": out["boxes"].detach().cpu(),
                "scores": out["scores"].detach().cpu(),
                "labels": out["labels"].detach().cpu(),
            })
            # Ground truth
            gts.append({
                "boxes": tgt["boxes"].detach().cpu(),
                "labels": tgt["labels"].detach().cpu(),
            })

        # Update metric batch by batch
        metric.update(preds, gts)

# After iterating through the test_loader, compute mAP
results = metric.compute()

print("mAP@0.5:", float(results["map_50"]))

Model loaded on: cpu


/usr/local/lib/python3.11/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


mAP@0.5: 0.6616380214691162


### Pruning 20% kênh cho SSD MobileNetV3 bằng `torch-pruning`

Trong bước này, mô hình SSD MobileNetV3 đã finetune được **pruning theo kiểu structured (cắt bớt kênh)** với tỉ lệ:

> `pruning_ratio = 0.2`  (tức là cắt ~20% kênh theo kiểu global)

#### Vì sao chỉ pruning 20%?

- Khi tăng tỉ lệ pruning lên **trên 20–30%**, số lượng kênh trong backbone bị cắt giảm mạnh, làm:
  - **Giảm năng lực biểu diễn** của backbone MobileNetV3.
  - Ảnh hưởng trực tiếp tới chất lượng đặc trưng truyền sang SSD head.
  - Đặc biệt với bài toán **object detection** trên dữ liệu:
    - Class imbalance (xe máy, xe hơi nhiều; xe tải, xe buýt ít),
    - Nhiều object nhỏ so với background,
    
  thì mô hình **rất nhạy với việc cắt kênh quá mạnh** → mAP tụt nhanh.

- Thực nghiệm (hoặc kinh nghiệm thực tế với các mô hình nhẹ) cho thấy:
  - Khi **pruning > 20%**, mAP thường giảm quá **2%** so với mô hình gốc,
  - Dù có finetune lại sau pruning, việc **phục hồi độ chính xác** trở nên khó hơn rất nhiều.

Vì vậy, **chọn tỉ lệ 20%** là một mức **cân bằng giữa:**

- **Giảm số params / FLOPs** một cách đáng kể,
- Nhưng vẫn giữ cấu trúc backbone đủ “giàu” để:
  - Không làm hỏng hoàn toàn feature map,
  - mAP sau finetune chỉ giảm trong ngưỡng chấp nhận được (**< 2%**).

Nói ngắn gọn:  
> Pruning 20% là một mức **an toàn** cho SSD MobileNetV3 trong bối cảnh bài toán này,  
> vừa giảm kích thước mô hình, vừa giúp việc phục hồi mAP sau finetune khả thi.

---

### Cách hoạt động của thuật toán pruning (torch-pruning)

Trong code, ta sử dụng `torch-pruning` với các thành phần chính:

1. **Phân tích Dependency Graph của mô hình**

   - `torch-pruning` xây dựng một **đồ thị phụ thuộc (Dependency Graph)** dựa trên:
     - Dòng chảy tensor giữa các layer,
     - Các kết nối đặc biệt (skip-connection, branch, depthwise + pointwise, v.v.).
   - Nhờ đồ thị này, khi cắt kênh ở một `Conv2d`, thư viện biết:
     - Cần cắt thêm kênh tương ứng ở các layer liên quan (BN, Conv kế tiếp, nhánh merge, …),
     - Tránh làm **“vỡ” kiến trúc** (in/out channels lệch nhau).

2. **Chọn độ quan trọng kênh bằng GroupMagnitudeImportance**

   - `importance = tp.importance.GroupMagnitudeImportance(p=1)`:
     - Tính độ quan trọng **theo norm L1** của nhóm tham số (channel/filter),
     - Kênh nào có norm nhỏ → ít đóng góp → **ưu tiên bị cắt**.
   - Vì dùng **global_pruning = True**, việc chọn kênh bị cắt:
     - Không làm theo từng layer độc lập,
     - Mà **xét toàn bộ mô hình**, chọn ra 20% kênh ít quan trọng nhất toàn mạng.

3. **Pruner: GroupNormPruner**

   - `GroupNormPruner` (dù tên là GroupNorm) thực ra là một pruner **group-based**, hỗ trợ:
     - Pruning theo nhóm kênh, đảm bảo consistency giữa Conv/BN/Linear liên quan,
     - Rất phù hợp với các kiến trúc **MobileNet/SSDLite** vốn nhiều depthwise + pointwise conv.
   - Ta truyền các tham số:
     - `iterative_steps=1`: prune 1 lần “mạnh” với tỉ lệ 20% (one-shot).
     - `pruning_ratio=0.2`: tỉ lệ kênh bị cắt.
     - `global_pruning=True`: chọn 20% kênh yếu nhất toàn mạng (global).

4. **Không đụng vào SSD head**

   - Trong code, `ignored_layers.append(model.head)`:
     - Chỉ prune phần **backbone / feature extractor**,
     - **Không prune SSD head** (các cls/reg prediction head).
   - Lý do:
     - Head rất nhạy cảm với số channel và số anchor,
     - Dễ gây lỗi shape hoặc làm hỏng hoàn toàn chất lượng dự đoán nếu chỉnh sai.

5. **Prune xong → test forward → lưu model**

   - Sau khi `pruner.step()`:
     - Cấu trúc mô hình đã được **tái tạo** với số kênh mới.
   - Chạy thử một lần forward với `example_inputs` để:
     - Kiểm tra model vẫn chạy được,
     - In ra số params trước và sau pruning.
   - Cuối cùng lưu luôn **full model**:
     - `torch.save(model, f"{model_name}_pruned_full.pth")`,
     - Vì kiến trúc đã thay đổi nên **state_dict đơn thuần không đủ** nếu không tái định nghĩa kiến trúc y hệt.

6. **Finetune lại mô hình đã prune**

   - Sau pruning, mô hình **mất một phần năng lực biểu diễn** → cần:
     - Train tiếp vài epoch (ở đây là ~20 epoch, LR nhỏ hơn),
     - Để “tái học” và phục hồi mAP về gần mức ban đầu.
   - Đây là bước rất quan trọng để:
     - Mô hình pruned **không bị drop mAP quá sâu**,
     - Đảm bảo tiêu chí **tụt < 2%** so với model full.


In [ ]:
import torch
import torch.nn as nn
import torch_pruning as tp
from torchvision.models.detection.ssdlite import ssdlite320_mobilenet_v3_large


def my_prune(model, example_inputs, output_transform=None, model_name="ssdlite320_mobilenet_v3_large"):
    """
    Function to perform structured pruning on a detection model (SSD MobileNetV3)
    using the torch-pruning library.

    Actions:
    - Build pruner based on GroupNormPruner + GroupMagnitudeImportance.
    - Globally prune 20% of channels across the entire backbone.
    - Ignore (do not prune) the SSD head.
    - Test one forward pass after pruning.
    - Save the full pruned model.
    """

    # (Import some classes for torch-pruning to handle if the model uses them)
    from torchvision.models.vision_transformer import VisionTransformer
    from torchvision.models.convnext import CNBlock, ConvNeXt

    # Count initial parameters
    ori_size = tp.utils.count_params(model)

    # Move model to CPU for safe graph tracing
    model.cpu().eval()

    # Enable gradients for all parameters (needed for certain pruner types)
    for p in model.parameters():
        p.requires_grad_(True)

    #########################################
    # 1. Define Ignored Layers (Do Not Prune)
    #########################################
    ignored_layers = []

    # For SSD model, ignore the head part (cls + bbox heads)
    if 'ssd' in model_name:
        ignored_layers.append(model.head)  # do not touch SSD head

    # Other setup for the pruner
    round_to = None
    channel_groups = {}
    unwrapped_parameters = None

    #########################################
    # 2. Initialize Pruner with GroupNormPruner
    #########################################
    # Importance: GroupMagnitudeImportance (L1-norm across channel groups)
    importance = tp.importance.GroupMagnitudeImportance(p=1)

    pruner = tp.pruner.GroupNormPruner(
        model,
        example_inputs=example_inputs,  # example input to build the dependency graph
        importance=importance,
        iterative_steps=1,              # one-shot pruning
        pruning_ratio=0.2,              # cut 20% of channels (global)
        global_pruning=True,            # select least important channels across the entire network
        round_to=round_to,
        unwrapped_parameters=unwrapped_parameters,
        ignored_layers=ignored_layers,  # do not prune SSD head
        channel_groups=channel_groups,
    )

    print(f"Model Name: {model_name}")
    tp.utils.print_tool.before_pruning(model)  # print initial parameter count

    # Save current channel configuration (optional, for logging)
    layer_channel_cfg = {}
    for module in model.modules():
        if module not in pruner.ignored_layers:
            if isinstance(module, nn.Conv2d):
                layer_channel_cfg[module] = module.out_channels
            elif isinstance(module, nn.Linear):
                layer_channel_cfg[module] = module.out_features

    #########################################
    # 3. Execute PRUNE
    #########################################
    # interactive=True -> returns each pruning group
    for g in pruner.step(interactive=True):
        g.prune()
    # or use the shorthand: pruner.step()

    # If the pruner has a regularizer, update and apply it to the model
    if isinstance(pruner, (tp.pruner.BNScalePruner,
                           tp.pruner.GroupNormPruner,
                           tp.pruner.GrowingRegPruner)):
        pruner.update_regularizer()
        pruner.regularize(model)

    tp.utils.print_tool.after_pruning(model)  # print parameter count after pruning

    #########################################
    # 4. SAVE PRUNED MODEL
    #########################################
    # Since the architecture has changed (new channel counts), we must:
    # => SAVE THE ENTIRE model (including architecture + weights)
    torch.save(model, f"{model_name}_pruned_full.pth")
    # A simple state_dict would be hard to use without redefining the exact new architecture.

    #########################################
    # 5. TEST ONE FORWARD PASS AFTER PRUNING
    #########################################
    model.eval()
    with torch.no_grad():
        if isinstance(example_inputs, dict):
            out = model(**example_inputs)
        else:
            # SSD takes input as list[Tensor], so example_inputs = [tensor]
            out = model(example_inputs)

        if output_transform is not None:
            out = output_transform(out)

        print(f"{model_name} Pruning:")
        params_after_prune = tp.utils.count_params(model)
        print("   Params: {} => {}".format(ori_size, params_after_prune))

        # Print output shape to ensure the model runs correctly
        if isinstance(out, (dict, list, tuple)):
            print("   Output:")
            for o in tp.utils.flatten_as_list(out):
                print("    ", o.shape)
        else:
            print("   Output:", out.shape)
        print("------------------------------------------------------\n")

    return model


if __name__ == "__main__":
    model_name = "ssdlite320_mbv3_finetuned"

    # PATH TO THE FINETUNED FULL MODEL
    FINETUNED_FULL_PATH = "/kaggle/working/ssdlite320_mbv3_finetuned_full.pth"

    # Select device (move back to GPU/CPU after pruning)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load the trained full model (structure + weights) onto CPU
    model = torch.load(FINETUNED_FULL_PATH, map_location="cpu", weights_only=False)
    model.eval()

    # SSD takes input as list[Tensor[C,H,W]] with size 320x320
    # -> Example input used to build DepGraph for torch-pruning
    example_inputs = [torch.randn(1, 3, 320, 320)] # Changed to batch size 1

    # Execute pruning
    pruned_model = my_prune(
        model,
        example_inputs=example_inputs,
        output_transform=None,
        model_name=model_name,
    )

    # Move the pruned model to the target device (CPU/GPU) for finetuning
    pruned_model = pruned_model.to(device)

    # --------------------------------------------------------
    # FINETUNE THE MODEL AFTER PRUNING
    # --------------------------------------------------------
    optimizer = torch.optim.Adam(
        [p for p in pruned_model.parameters() if p.requires_grad],
        lr=1e-4,                      # slightly smaller than original training to avoid destroying weights
        weight_decay=5e-4,
    )
    
    num_epochs = 20  # a few epochs to recover mAP after pruning
    
    # NOTE: Assuming train_loader is defined and accessible here from previous snippets
    try:
        if 'train_loader' not in globals():
            # Mock DataLoader if not available for this standalone script execution
            print("Warning: train_loader not found. Skipping finetuning loop.")
            train_loader = []
    except NameError:
        print("Warning: train_loader not found. Skipping finetuning loop.")
        train_loader = []

    for epoch in range(num_epochs):
        pruned_model.train()
        running_loss = 0.0
    
        if not train_loader:
             break # Exit finetuning if loader is unavailable

        for images, targets in train_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
    
            optimizer.zero_grad()
            loss_dict = pruned_model(images, targets)
            loss = sum(loss for loss in loss_dict.values())
            loss.backward()
            optimizer.step()
    
            running_loss += loss.item()
        
        if len(train_loader) > 0:
            print(f"[Prune FT Epoch {epoch+1}/{num_epochs}] loss = {running_loss/len(train_loader):.4f}")
    
    # Save the model after pruning + finetuning
    torch.save(pruned_model.state_dict(), "ssdlite320_mbv3_pruned_finetuned_state.pth")
    torch.save(pruned_model, "ssdlite320_mbv3_pruned_finetuned_full.pth")
    print("Finished pruning", model_name)

Model Name: ssdlite320_mbv3_finetuned
SSD(
  (backbone): SSDLiteFeatureExtractorMobileNet(
    (features): Sequential(
      (0): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False) => (0): Conv2d(3, 7, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True) => (1): BatchNorm2d(7, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (2): Hardswish()
        )
        (1): InvertedResidual(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False) => (0): Conv2d(7, 7, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=7, bias=False)
              (1): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True) => (1): BatchNorm2d(7, 

### Đánh giá lại mAP@0.5 sau khi pruning và finetune

Sau khi mô hình SSD MobileNetV3 được **pruning 50% kênh** và **finetune thêm 20 epoch**, bước cuối cùng là **đánh giá lại chất lượng** mô hình trên tập test.

Cách đánh giá tương tự như trước pruning:

- Load lại mô hình pruned (`*_pruned_finetuned_full.pth`)
- Sử dụng `MeanAveragePrecision` từ TorchMetrics với `iou_thresholds = [0.5]`
- Chạy qua `test_loader` và cập nhật metric cho từng batch

Điểm quan trọng của bước này:

#### 1. Kiểm tra mức giảm mAP sau pruning  
Mục tiêu đề ra là:

> **mAP sau pruning không giảm quá 2% so với mô hình gốc**

Việc đánh giá lại giúp chúng ta xác nhận:

- Pruning 50% là mức **an toàn**  
- Finetune sau pruning đã giúp mô hình **khôi phục năng lực dự đoán**, đặc biệt cho các object nhỏ hoặc lớp xuất hiện ít

#### 2. Kiểm tra tính toàn vẹn của kiến trúc  
Sử dụng `torchmetrics` đảm bảo:

- Tensor output có dạng `{boxes, labels, scores}`
- Không có lỗi shape do cắt kênh sai hoặc mismatch giữa backbone và SSD head
- Model hoàn toàn **chạy ổn định** sau khi pruning

#### 3. Đánh giá hiệu quả trade-off  
Sau pruning, mô hình:

- Nhẹ hơn  
- Ít FLOPs hơn  
- Nhưng độ chính xác phải giữ được mức khả thi (≈ giống trước khi prune)

Kết quả mAP@0.5 từ đoạn code này là thước đo cuối cùng để xem mô hình sau prune còn đáp ứng yêu cầu của bài toán hay không.


In [ ]:
import torch
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# NOTE: test_loader must be defined and accessible in the execution environment.

device = torch.device("cpu")  # Force to CPU to avoid anchor / ops GPU errors

# ------------------------------------------------------------
# Load the pruned + finetuned model
# ------------------------------------------------------------
model = torch.load(
    "/kaggle/working/ssdlite320_mbv3_pruned_finetuned_full.pth",
    map_location=device,
    weights_only=False,  # Important for PyTorch >= 2.6
)

model.to(device)
model.eval()
print("Pruned Model loaded on:", device)

# ------------------------------------------------------------
# mAP@0.5 Metric
# ------------------------------------------------------------
metric = MeanAveragePrecision(
    iou_type="bbox",
    iou_thresholds=[0.5],  # Only calculate mAP@0.5
)

# ------------------------------------------------------------
# Evaluate the entire test_loader
# ------------------------------------------------------------
# NOTE: The test_loader variable must be defined in the environment for this loop to run.
if 'test_loader' not in globals():
    print("Warning: 'test_loader' is not defined. Skipping evaluation loop.")
else:
    with torch.no_grad():
        for images, targets in test_loader:
            # images is list[Tensor[C,H,W]]
            images = [img.to(device) for img in images]

            # Run inference
            outputs = model(images)

            # Convert outputs and targets to the format required by torchmetrics
            preds, gts = [], []

            for out, tgt in zip(outputs, targets):
                preds.append({
                    "boxes":  out["boxes"].detach().cpu(),
                    "scores": out["scores"].detach().cpu(),
                    "labels": out["labels"].detach().cpu(),
                })
                gts.append({
                    "boxes":  tgt["boxes"].detach().cpu(),
                    "labels": tgt["labels"].detach().cpu(),
                })

            metric.update(preds, gts)

    # ------------------------------------------------------------
    # Compute and print mAP after iterating through the entire test set
    # ------------------------------------------------------------
    results = metric.compute()
    print(" mAP@0.5 (after pruning):", float(results["map_50"]))

Pruned Model loaded on: cpu
📊 mAP@0.5 (after pruning): 0.6371687650680542


### Quantization INT8 cho mô hình Object Detection (PyTorch → ONNX → INT8)

Bước cuối cùng của pipeline là giảm kích thước mô hình và (về lý thuyết) tăng tốc suy luận bằng cách chuyển mô hình **float32 → INT8**.  
Tuy nhiên, cả TensorFlow và PyTorch đều **không hỗ trợ tốt quantization** cho các mô hình **Object Detection**, đặc biệt là **QAT (Quantization Aware Training)**. Vì vậy, ta phải dùng hướng **PyTorch → ONNX → ONNX INT8**.

---

## 1. Vì sao TensorFlow và PyTorch không hỗ trợ tốt QAT cho Object Detection?

### 1.1 TensorFlow / TFLite

TFLite hỗ trợ quantization rất tốt cho classification (MobileNet, EfficientNet, ResNet…), nhưng **pipeline detection** lại có nhiều hạn chế:

- Kiến trúc detection như SSD, Faster R-CNN, YOLO… chứa **nhiều nhánh phức tạp**: backbone, FPN, classification head, regression head, decode boxes, NMS.
- Nhiều op quan trọng **không có kernel INT8** trong TFLite (hoặc hỗ trợ hạn chế).
- QAT yêu cầu toàn bộ model hỗ trợ INT8 → không thỏa mãn cho hầu hết detection model.

Kết quả:  
> Dù huấn luyện QAT, mô hình detection thường không thể chạy full INT8 thực sự trong TFLite.

---

### 1.2 PyTorch

PyTorch hỗ trợ 3 loại quantization:

- **Dynamic Quantization**
- **Static Quantization**
- **QAT**

Nhưng các API này **chỉ vận hành tốt** với:

- CNN classification đơn giản  
- MLP  
- LSTM / Transformer cơ bản  

Với mô hình detection như **SSDLite MobileNetV3 (torchvision)**:

- Input/output là `list[Tensor]`, `list[dict]` → khó trace để quantize.
- Có nhiều module đặc thù detection (`AnchorGenerator`, `SSDHead`, BoxCoder, NMS...).
- Không có pipeline quantization chính thức trong `torchvision`.

Do đó:

> PyTorch **không hỗ trợ QAT hoặc static quantization** cho SSD một cách trực tiếp.  
> Nếu muốn làm, phải tự xây lại toàn bộ detection pipeline → quá phức tạp trong phạm vi đồ án.

---

## 2. Vì sao phải convert sang ONNX để quantize?

Do PyTorch không hỗ trợ tốt quantization detection → giải pháp hợp lý là:

1. Huấn luyện mô hình trong PyTorch  
2. **Export sang ONNX FP32**  
3. **Dùng ONNX Runtime để quantize trọng số**

Ưu điểm:

- ONNX là chuẩn chung, dễ triển khai lên nhiều thiết bị edge.
- `onnxruntime.quantization` hỗ trợ nhiều phương pháp quantization sau huấn luyện (PTQ).

Trong code, bạn làm:

```python
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    ONNX_FP32_PATH,
    ONNX_INT8_PATH,
    weight_type=QuantType.QInt8,  # chỉ quant trọng số
)
```

Điều quan trọng cần nhấn mạnh:

### Đây là **dynamic quantization (weight-only)**  
- Chỉ quant **trọng số (weights)** của các layer phù hợp (thường là Linear, đôi lúc là Conv).
- **Activations vẫn là float32.**
- Không cần tập calibration.
- Không cần QAT.

Ưu điểm:

- Dễ dùng, ít rủi ro.
- Không phải sửa kiến trúc detection.
- File ONNX nhỏ hơn.

Nhược điểm:

- Không phải INT8 “full” (weights + activations).
- Tăng tốc không nhiều nếu CPU không hỗ trợ INT8 tốt.

---

## 3. Tại sao ONNX INT8 không chạy được trên Kaggle?

Khi chạy ONNX INT8, ONNXRuntime cần kernel để chạy **ConvInteger** (phiên bản INT8 của Conv2d).  
Nhưng CPU trên Kaggle (Xeon v3/v4) **không hỗ trợ** các tập lệnh INT8 hiện đại như:

- AVX512 VNNI  
- AMX  
- AVX512-VBMI  
- Tensor Cores INT8  
- ARM NEON dot-product  

Kết quả:

- ONNXRuntime không tìm được implementation cho node `ConvInteger` → lỗi:

  ```
  NotImplemented: no implementation for ConvInteger(...)
  ```

=> **Trên Kaggle không thể chạy mô hình Conv-INT8**.  
Chỉ có thể load và chạy ONNX FP32.  
Benchmark INT8 phải làm trên thiết bị đích như Jetson (TensorRT), CPU mới, NPU…

---

## 4. Không có QAT thì mAP có tụt không?

### Có – nhưng mức độ tùy phương pháp quantization.

Trong bài này, ta dùng **dynamic quantization, weight-only**:

- Chỉ quant trọng số  
- Activations vẫn float32  
- Không có bước train lại như QAT  

Do đó:

- Sai số ít hơn static full INT8 (weights + activations)
- mAP thường giảm **nhẹ**, vài phần trăm → điều này **bình thường**
- Với SSD (nhạy với regression + object nhỏ), việc tụt mAP là hiển nhiên

Nhưng:

> Mục tiêu của đồ án là hoàn thiện pipeline finetune → prune → export ONNX → quantize INT8,  
> không tối ưu hóa mAP INT8, nên mức suy giảm nhỏ là chấp nhận được.

---

### Tổng kết

- PyTorch & TensorFlow **không hỗ trợ QAT** cho SSD MobileNetV3 → phải dùng ONNX.  
- ONNX dynamic quant (weight-only) giúp giảm kích thước model mà không cần QAT.  
- ONNX INT8 không chạy được trên Kaggle vì không có kernel ConvInteger.  
- Nếu cần benchmark tốc độ INT8 → phải làm trên thiết bị hỗ trợ INT8 (Jetson, CPU mới, NPU).  


In [17]:
!pip install onnx onnxruntime onnxruntime-tools -q

In [ ]:
import torch
import torch.nn as nn
import torch.onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

# ================== CONFIG ==================
MODEL_FLOAT_PATH   = "/kaggle/working/ssdlite320_mbv3_pruned_finetuned_full.pth"
ONNX_FP32_PATH     = "/kaggle/working/ssd_pruned_fp32.onnx"
ONNX_INT8_PATH     = "/kaggle/working/ssd_pruned_int8.onnx"

device = torch.device("cpu")
print("Device:", device)

# ============ 1. LOAD PYTORCH MODEL (PRUNED + FINETUNE) ============
model = torch.load(
    MODEL_FLOAT_PATH,
    map_location=device,
    weights_only=False,  # because full model was saved
)
model.to(device).eval()
print("Loaded pruned finetuned PyTorch model float32.")

# ============ 2. ONNX WRAPPER (batch size = 1) ============
# ONNX only works with Tensor, not list[dict] so we wrap it
class SSDExportWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        # x shape: (1, 3, 320, 320)
        images = [x[0]]               # list[Tensor[3,320,320]]
        outputs = self.model(images)  # list[dict]
        out = outputs[0]
        # ONNX output: 3 tensors (boxes, scores, labels)
        return out["boxes"], out["scores"], out["labels"]

wrapper = SSDExportWrapper(model).to(device)

# Create a dummy input tensor for tracing
dummy = torch.randn(1, 3, 320, 320, device=device)

# ============ 3. EXPORT ONNX FP32 ============
torch.onnx.export(
    wrapper,
    dummy,
    ONNX_FP32_PATH,
    input_names=["input"],
    output_names=["boxes", "scores", "labels"],
    opset_version=12,
    dynamic_axes=None,  # fixed batch=1 for simplicity
)

print(f"Saved FP32 ONNX to: {ONNX_FP32_PATH}")

# ============ 4. QUANTIZE ONNX -> INT8 (WEIGHT) ============

quantize_dynamic(
    ONNX_FP32_PATH,
    ONNX_INT8_PATH,
    weight_type=QuantType.QInt8,  # quantize weights, activations remain float
)

print(f"Saved INT8 ONNX (dynamic quant) to: {ONNX_INT8_PATH}")

Device: cpu
✅ Loaded pruned + finetuned PyTorch model (float32).


  elem_type: 7
  shape {
    dim {
      dim_param: "unk__7"
    }
    dim {
      dim_value: 2
    }
  }
}
.


💾 Saved FP32 ONNX to: /kaggle/working/ssd_pruned_fp32.onnx


  elem_type: 7
  shape {
    dim {
      dim_param: "unk__246"
    }
    dim {
      dim_param: "unk__247"
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__248"
    }
    dim {
      dim_param: "unk__249"
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__252"
    }
    dim {
      dim_param: "unk__253"
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__254"
    }
    dim {
      dim_param: "unk__255"
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__258"
    }
    dim {
      dim_param: "unk__259"
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__260"
    }
    dim {
      dim_param: "unk__261"
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__264"
    }
    dim {
      dim_param: "unk__265"
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_param: "unk__266"
    }
    dim {
      dim_param: "unk__267"
    }
  }
}
.
  elem_type: 7
  shape {
    dim

💾 Saved INT8 ONNX (dynamic quant) to: /kaggle/working/ssd_pruned_int8.onnx
